In [4]:
## Executado no Colab (A100)

In [ ]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import torch
import gc
import shutil

from pathlib import Path


In [2]:
df = pd.read_parquet("/content/drive/MyDrive/Mestrado/Artigo KDMILE/df_completo.parquet")

In [ ]:
from google.colab import drive

drive.flush_and_unmount()

drive.mount('/content/drive')

In [ ]:
import os
import shutil
import gc
from pathlib import Path
from huggingface_hub import snapshot_download, login
from google.colab import drive
from google.colab import userdata

# =========================
# 1. SETUP
# =========================

login(userdata.get('HF_TOKEN'))

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

DRIVE_DIR = Path("/content/drive/MyDrive/Modelos_HF")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_DIR = Path("/content/models")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = Path("/content/hf_cache")

models = [
    "ICT-TIME-and-Querit/ICT-TIME-and-Querit-embedding-v1"
    "PORTULAN/serafim-335m-portuguese-pt-sentence-encoder",
    "PORTULAN/serafim-335m-portuguese-pt-sentence-encoder-ir",
    "josedossantos/bertimbau-tuned"
    "codefuse-ai/F2LLM-v2-14B",
    "joaorobson/harrier-oss-v1-27b", #
    "Octen/Octen-Embedding-8B",
    "joaorobson/KaLM-Embedding-Gemma3-12B-2511", #
    "Qwen/Qwen3-Embedding-8B",
    "jinaai/jina-embeddings-v5-text-small",
    "nvidia/llama-embed-nemotron-8b",
]

# =========================
# 2. LOOP STREAMING (1 POR VEZ)
# =========================

for model_id in models:
    print(f"\n🚀 Baixando: {model_id}")

    model_name = model_id.split("/")[-1]
    local_path = LOCAL_DIR / model_name
    drive_path = DRIVE_DIR / model_name

    try:
        print(drive_path)
        if drive_path.exists():
            print("⚠ Já existe no Drive...")
            continue
        # =========================
        # DOWNLOAD (SSD /content)
        # =========================

        snapshot_download(
            repo_id=model_id,
            local_dir=str(drive_path),
            local_dir_use_symlinks=False,
        )

        print(f"✔ Download concluído: {model_name}")

        # =========================
        # COPIA PARA DRIVE
        # =========================

        print("📤 Copiando para Drive...")
        #shutil.copytree(local_path, drive_path)
        print("✔ Copiado para Drive")

    except Exception as e:
        print(f"❌ Erro em {model_id}: {e}")

    # =========================
    # 🔥 LIMPEZA TOTAL (CRÍTICO)
    # =========================

    print("🧹 Limpando /content para liberar espaço...")

    if local_path.exists():
        shutil.rmtree(local_path, ignore_errors=True)

    if CACHE_DIR.exists():
        shutil.rmtree(CACHE_DIR, ignore_errors=True)

    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    gc.collect()

    # DEBUG DE ESPAÇO
    print("✔ Limpeza concluída")
    print("📊 Espaço atual em /content:")
    os.system("df -h /content")

print("\n🎉 TODOS OS MODELOS PROCESSADOS COM SUCESSO!")

In [ ]:
!pip install --upgrade torchao

In [ ]:
gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    try:
        torch.cuda.synchronize()
    except Exception:
        pass


## Embeddings multilíngues

In [ ]:
import gc
import torch
import pandas as pd

from pathlib import Path
from sentence_transformers import SentenceTransformer

EMBEDDING_CONFIGS = [
    {
        "model_name": "ICT-TIME-and-Querit/ICT-TIME-and-Querit-embedding-v1",
        "task": None,
        "suffix": ""
    },
    {
        "model_name": "codefuse-ai/F2LLM-v2-14B",
        "task": None,
        "suffix": ""
    },
    {
        "model_name": "joaorobson/harrier-oss-v1-27b",
        "task": None,
        "suffix": ""
    },
    {
        "model_name": "Octen/Octen-Embedding-8B",
        "task": None,
        "suffix": ""
    },
    {
        "model_name": "joaorobson/KaLM-Embedding-Gemma3-12B-2511",
        "task": None,
        "suffix": ""
    },
    {
        "model_name": "Qwen/Qwen3-Embedding-8B",
        "task": None,
        "suffix": ""
    },
    {
        "model_name": "jinaai/jina-embeddings-v5-text-small",
        "task": "clustering",
        "suffix": "__clustering"
    },
    {
        "model_name": "jinaai/jina-embeddings-v5-text-small",
        "task": "text-matching",
        "suffix": "__text_matching"
    },
    {
        "model_name": "nvidia/llama-embed-nemotron-8b",
        "task": None,
        "suffix": ""
    },
]

CPU_MODELS = {
              }

TEXT_COLUMNS = [
    "texto",
    "texto_preprocessado",
    "texto_preprocessado_sem_justificativa",
]

MODELS_DIR = Path("/content/drive/MyDrive/Modelos_HF")

OUTPUT_FILE = Path(
    "/content/drive/MyDrive/Mestrado/Artigo KDMILE/embeddings.parquet"
)

# ------------------------------------------------------------------
# Carrega parquet existente
# ------------------------------------------------------------------

if OUTPUT_FILE.exists():
    df = pd.read_parquet(OUTPUT_FILE)
    print(f"Loaded existing parquet: {OUTPUT_FILE}")
else:
    raise FileNotFoundError(
        f"Parquet not found: {OUTPUT_FILE}\n"
        "Load/create your dataframe before running."
    )

# ------------------------------------------------------------------
# Função para verificar se embedding já existe e está preenchido
# ------------------------------------------------------------------

def embedding_complete(df, column_name):

    if column_name not in df.columns:
        return False

    serie = df[column_name]

    if len(serie) != len(df):
        return False

    if serie.isna().any():
        return False

    try:
        sample = serie.iloc[0]

        if sample is None:
            return False

        if not hasattr(sample, "__len__"):
            return False

        if len(sample) == 0:
            return False

    except Exception:
        return False

    return True


# ------------------------------------------------------------------
# Loop principal
# ------------------------------------------------------------------

for cfg in EMBEDDING_CONFIGS:

    model_name = cfg["model_name"]
    task_name = cfg["task"]
    suffix = cfg["suffix"]

    print("\n" + "=" * 80)
    print(f"Model: {model_name}")

    model_slug = (
      model_name
      .replace("/", "__")
      .replace("-", "_")
      .replace(".", "_")
  ) + suffix

    # --------------------------------------------------------------
    # Descobre quais colunas ainda precisam ser geradas
    # --------------------------------------------------------------

    pending_columns = []

    for col in TEXT_COLUMNS:

        embedding_col = f"embedding__{model_slug}__{col}"

        if embedding_complete(df, embedding_col):

            print(f"[OK] {embedding_col}")

        else:

            pending_columns.append(col)

    # --------------------------------------------------------------
    # Nada a fazer para este modelo
    # --------------------------------------------------------------

    if not pending_columns:

        print(
            f"Skipping {model_name} "
            f"(all embeddings already generated)"
        )

        continue

    print(
        f"Pending columns: {pending_columns}"
    )

    # --------------------------------------------------------------
    # Verifica modelo local
    # --------------------------------------------------------------

    model_folder = model_name.split("/")[-1]

    local_model_path = MODELS_DIR / model_folder

    if not local_model_path.exists():

        print(f"Model not found: {local_model_path}")

        continue

    model = None

    try:

        print(f"Loading model from: {local_model_path}")
        if model_name in CPU_MODELS:
          model = SentenceTransformer(
            str(local_model_path),
            trust_remote_code=True,
            device="cpu",
            model_kwargs={
                "torch_dtype": torch.bfloat16,
            }
          )
        else:
          model = SentenceTransformer(
              str(local_model_path),
              trust_remote_code=True,
              device="cuda",
              model_kwargs={
                  "torch_dtype": torch.bfloat16
              }
          )

        model.max_seq_length = 32768

        # ----------------------------------------------------------
        # Gera apenas colunas pendentes
        # ----------------------------------------------------------

        for col in pending_columns:

            embedding_col = f"embedding__{model_slug}__{col}"

            print("\n" + "-" * 80)
            print(f"Generating: {embedding_col}")

            texts = (
                df[col]
                .fillna("")
                .astype(str)
                .tolist()
            )

            encode_kwargs = dict(
                sentences=texts,
                batch_size=4,
                show_progress_bar=True,
                normalize_embeddings=True,
                convert_to_numpy=True,
            )

            # Jina v5 suporta task específica
            if task_name is not None:
              encode_kwargs["task"] = task_name

            embeddings = model.encode(**encode_kwargs)

            print(
                f"Embeddings shape: {embeddings.shape}"
            )

            df[embedding_col] = list(embeddings)

            print(f"Saving parquet...")

            df.to_parquet(
                OUTPUT_FILE,
                index=False
            )

            print(f"Saved: {OUTPUT_FILE}")

    except Exception as e:

        print(f"\nERROR processing {model_name}")
        print(type(e).__name__)
        print(e)

    finally:

        print("\nCleaning memory...")

        if model is not None:
            del model

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()

            try:
                torch.cuda.synchronize()
            except Exception:
                pass

        gc.collect()

        print("Done.")

print("\nFinished!")

## Embeddings BERTimbau

In [ ]:
import gc
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from sentence_transformers import SentenceTransformer

# ============================================================
# CONFIG
# ============================================================

MODEL_CONFIGS = [
    {
        "model_name": "PORTULAN/serafim-335m-portuguese-pt-sentence-encoder",
        "strategy": "trunc128",
        "max_seq_length": 128,
        "chunk_tokens": 128,
    },
    {
        "model_name": "PORTULAN/serafim-335m-portuguese-pt-sentence-encoder",
        "strategy": "meanpool128",
        "max_seq_length": 128,
        "chunk_tokens": 128,
    },
    {
        "model_name": "PORTULAN/serafim-335m-portuguese-pt-sentence-encoder-ir",
        "strategy": "trunc128",
        "max_seq_length": 128,
        "chunk_tokens": 128,
    },
    {
        "model_name": "PORTULAN/serafim-335m-portuguese-pt-sentence-encoder-ir",
        "strategy": "meanpool128",
        "max_seq_length": 128,
        "chunk_tokens": 128,
    },
    {
        "model_name": "josedossantos/bertimbau-tuned",
        "strategy": "trunc512",
        "max_seq_length": 512,
        "chunk_tokens": 512,
    },
    {
        "model_name": "josedossantos/bertimbau-tuned",
        "strategy": "meanpool512",
        "max_seq_length": 512,
        "chunk_tokens": 512,
    },
]

TEXT_COLUMNS = [
    "texto",
    "texto_preprocessado",
    "texto_preprocessado_sem_justificativa",
]

MODELS_DIR = Path(
    "/content/drive/MyDrive/Modelos_HF"
)

OUTPUT_FILE = Path(
    "/content/drive/MyDrive/Mestrado/Artigo KDMILE/embeddings.parquet"
)

# ============================================================
# LOAD DATAFRAME
# ============================================================

if OUTPUT_FILE.exists():
    df = pd.read_parquet(OUTPUT_FILE)
    print(f"Loaded: {OUTPUT_FILE}")
else:
    raise FileNotFoundError(
        f"Parquet not found: {OUTPUT_FILE}"
    )

# ============================================================
# HELPERS
# ============================================================

def embedding_complete(df, column_name):

    if column_name not in df.columns:
        return False

    serie = df[column_name]

    if len(serie) != len(df):
        return False

    if serie.isna().any():
        return False

    try:
        sample = serie.iloc[0]

        if sample is None:
            return False

        if not hasattr(sample, "__len__"):
            return False

        if len(sample) == 0:
            return False

    except Exception:
        return False

    return True


def encode_chunk_mean(
    model,
    texts,
    chunk_tokens=128,
    batch_size=16,
):
    """
    Divide documentos longos em chunks
    e faz média dos embeddings.
    """

    tokenizer = model.tokenizer

    all_embeddings = []

    for idx, text in enumerate(texts):

        token_ids = tokenizer.encode(
            text,
            add_special_tokens=False
        )

        if len(token_ids) == 0:

            chunk_emb = model.encode(
                [""],
                convert_to_numpy=True,
                normalize_embeddings=False,
                show_progress_bar=False
            )[0]

            all_embeddings.append(chunk_emb)

            continue

        chunks = [
            token_ids[i:i + chunk_tokens]
            for i in range(
                0,
                len(token_ids),
                chunk_tokens
            )
        ]

        chunk_texts = [
            tokenizer.decode(
                chunk,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )
            for chunk in chunks
        ]

        chunk_embeddings = model.encode(
            chunk_texts,
            batch_size=batch_size,
            show_progress_bar=False,
            normalize_embeddings=False,
            convert_to_numpy=True,
        )

        doc_embedding = np.mean(
            chunk_embeddings,
            axis=0
        )

        norm = np.linalg.norm(
            doc_embedding
        )

        if norm > 0:
            doc_embedding = (
                doc_embedding / norm
            )

        all_embeddings.append(
            doc_embedding
        )

        if idx % 500 == 0:
            print(
                f"{idx}/{len(texts)}"
            )

    return np.vstack(
        all_embeddings
    )

# ============================================================
# MAIN LOOP
# ============================================================

for cfg in MODEL_CONFIGS:

    model_name = cfg["model_name"]
    strategy = cfg["strategy"]
    max_seq_length = cfg["max_seq_length"]
    chunk_tokens = cfg["chunk_tokens"]

    print("\n" + "=" * 80)
    print(model_name)
    print(strategy)

    model_slug = (
        model_name
        .replace("/", "__")
        .replace("-", "_")
        .replace(".", "_")
    )

    model_slug += f"__{strategy}"

    pending_columns = []

    for col in TEXT_COLUMNS:

        embedding_col = (
            f"embedding__{model_slug}__{col}"
        )

        if embedding_complete(
            df,
            embedding_col
        ):
            print(
                f"[OK] {embedding_col}"
            )
        else:
            pending_columns.append(
                col
            )

    if not pending_columns:

        print(
            "All embeddings already generated."
        )

        continue

    print(
        f"Pending columns: {pending_columns}"
    )

    model_folder = (
        model_name.split("/")[-1]
    )

    local_model_path = (
        MODELS_DIR / model_folder
    )

    if not local_model_path.exists():

        print(
            f"Model not found: "
            f"{local_model_path}"
        )

        continue

    model = None

    try:

        print(
            f"Loading model: "
            f"{local_model_path}"
        )

        model = SentenceTransformer(
            str(local_model_path),
            trust_remote_code=True,
            device="cuda",
            model_kwargs={
                "torch_dtype": torch.bfloat16
            }
        )

        print(
            "SentenceTransformer:",
            model.max_seq_length
        )

        print(
            "Tokenizer:",
            model.tokenizer.model_max_length
        )

        try:
            print(
                "Backbone:",
                model._first_module()
                .auto_model
                .config
                .max_position_embeddings
            )
        except:
            pass

        # ----------------------------------------------------
        # SERAFIM ORIGINAL
        # ----------------------------------------------------

        model.max_seq_length = max_seq_length

        for col in pending_columns:

            embedding_col = (
                f"embedding__{model_slug}__{col}"
            )

            print(
                "\n" + "-" * 80
            )

            print(
                f"Generating "
                f"{embedding_col}"
            )

            texts = (
                df[col]
                .fillna("")
                .astype(str)
                .tolist()
            )

            if strategy.startswith("trunc"):

              embeddings = model.encode(
                  texts,
                  batch_size=16,
                  show_progress_bar=True,
                  normalize_embeddings=False,
                  convert_to_numpy=True,
              )

              embeddings = embeddings / np.linalg.norm(
                  embeddings,
                  axis=1,
                  keepdims=True
              )
            elif strategy.startswith("meanpool"):

              embeddings = encode_chunk_mean(
                  model=model,
                  texts=texts,
                  chunk_tokens=chunk_tokens,
                  batch_size=16,
              )

            else:

                raise ValueError(
                    f"Unknown strategy: {strategy}"
                )

            print(
                "Shape:",
                embeddings.shape
            )

            df[
                embedding_col
            ] = list(
                embeddings
            )

            print(
                "Saving parquet..."
            )

            df.to_parquet(
                OUTPUT_FILE,
                index=False
            )

            print(
                f"Saved: "
                f"{OUTPUT_FILE}"
            )

    except Exception as e:

        print(
            f"\nERROR "
            f"{model_name}"
        )

        print(
            type(e).__name__
        )

        print(e)

    finally:

        print(
            "\nCleaning memory..."
        )

        if model is not None:
            del model

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()

            try:
                torch.cuda.synchronize()
            except:
                pass

        gc.collect()

        print("Done.")

print("\nFinished!")